# INFO 2950-204 Final Project: Data Cleaning Appendix

This notebook turns our saved raw API pulls into **`analysis_ready.csv`**, the main file for our purposes. We respond to the Phase 2 feedback in four ways..

1. **Primary performance measure is now Q1-based.** Each team's fastest valid Q1 lap is compared with the fastest valid Q1 lap of the session. The old whole-session gap to pole is kept as a secondary check, and **season averages are recalculated for both measures**.
2. **One stable team ID across rebrands** (`team_id`), so team dummies mean the same thing in every season.
3. **Wet-session flag matched to Q1** (`wet_q1`), plus Q1-window track temperature (`track_temp_q1`) from OpenF1's trackside sensors.
4. **Explicit rules** for missing laps, wet sessions and outliers, written out at the end of this notebook.

**Inputs** (saved in Phase 2 notebook, in [`data/raw/`](https://github.com/rbllrd/info2950-204_project/tree/main/data/raw)):

| File | Source | What it holds |
|---|---|---|
| `openf1_sessions.csv` | OpenF1 `/sessions` | every session, with start/end times |
| `openf1_drivers.csv` | OpenF1 `/drivers` | each driver's team **for each session** |
| `openf1_laps_qualifying.csv` | OpenF1 `/laps` | every qualifying lap |
| `openf1_weather_qualifying.csv` | OpenF1 `/weather` | trackside readings, about once a minute |
| `openmeteo_hourly_qualifying.csv` | Open-Meteo archive | hourly air temperature and cloud cover at session start |
| `openf1_session_result_qualifying.csv` | OpenF1 `/session_result` | **new:** official Q1/Q2/Q3 times per driver |

**Output:** `analysis_ready.csv` (one row = one team in one qualifying session).

## 1. Setup

In [1]:
#libraries
import os
import json
import time
from urllib.request import urlopen
from urllib.error import HTTPError

import pandas as pd # dataframes
import duckdb # SQL on dataframes

# keep printed tables narrow so they are not cut off in the PDF export
pd.set_option('display.width', 100)
pd.set_option('display.max_columns', 10)
pd.set_option('display.precision', 3)

RAW = 'data/raw' # folder holding the saved raw API pulls

We import our libraries and set pandas to print narrow tables. The Phase 2 PDF cut off wide tables, so every table below shows only the columns needed for that step.

In [2]:
# helper: parse a date column and express it as naive EST, matching the rest of our data
def to_est(col):
    t = pd.to_datetime(col, format='mixed')
    if t.dt.tz is not None: # timestamps with a UTC offset -> convert to EST
        t = t.dt.tz_convert('EST').dt.tz_localize(None)
    return t # timestamps without an offset were already saved in EST

OpenF1 stores times in UTC, but our Phase 2 notebook converted some columns to EST before saving them. This helper converts anything that still has a UTC offset to EST and leaves already-converted columns alone, so every time in this notebook is on the same clock (fixed EST, UTC−5).

## 2. Load the saved raw data

In [3]:
# load each raw API pull saved in data/raw/
temp_sessions_df = pd.read_csv(f'{RAW}/openf1_sessions.csv')
temp_drivers_df = pd.read_csv(f'{RAW}/openf1_drivers.csv')
temp_laps_df = pd.read_csv(f'{RAW}/openf1_laps_qualifying.csv')
temp_f1_weather_df = pd.read_csv(f'{RAW}/openf1_weather_qualifying.csv')
temp_openmeteo_df = pd.read_csv(f'{RAW}/openmeteo_hourly_qualifying.csv')

for name, d in [('sessions', temp_sessions_df), ('drivers', temp_drivers_df), ('laps', temp_laps_df),
                ('OpenF1 weather', temp_f1_weather_df), ('Open-Meteo', temp_openmeteo_df)]:
    print(f'{name:15} {d.shape[0]:>7,} rows x {d.shape[1]} columns')

sessions            619 rows x 15 columns
drivers          10,059 rows x 12 columns
laps             26,338 rows x 16 columns
OpenF1 weather    7,176 rows x 10 columns
Open-Meteo           88 rows x 14 columns


We read the five raw files our Phase 2 notebook saved instead of calling the APIs again. This makes the cleaning reproducible and fast; the collection code that produced these files is in our Phase 2 notebook.

## 3. Qualifying sessions

In [4]:
# keep qualifying sessions (main and sprint) from completed seasons that were not cancelled
quali_df = duckdb.sql('''
    SELECT session_key, session_name, date_start, date_end, circuit_short_name, year
    FROM temp_sessions_df
    WHERE session_type = 'Qualifying' AND is_cancelled = False AND year < 2026
''').df()
quali_df['date_start'] = to_est(quali_df['date_start'])
quali_df['date_end'] = to_est(quali_df['date_end'])
quali_df['is_sprint'] = quali_df['session_name'] != 'Qualifying' # Sprint Shootout / Sprint Qualifying

print(len(quali_df), 'qualifying sessions;', quali_df['is_sprint'].sum(), 'are sprint qualifying')
quali_df[['session_key', 'session_name', 'circuit_short_name', 'date_start']].head()

88 qualifying sessions; 18 are sprint qualifying


,session_key,session_name,circuit_short_name,date_start
0,7768,Qualifying,Sakhir,2023-03-04 10:00:00
1,7775,Qualifying,Jeddah,2023-03-18 12:00:00
2,7783,Qualifying,Melbourne,2023-04-01 01:00:00
3,9064,Qualifying,Baku,2023-04-28 08:00:00
4,9278,Sprint Qualifying,Baku,2023-04-29 03:30:00


We keep every qualifying session from 2023–2025 that was not cancelled. Sprint qualifying sessions are kept but flagged with `is_sprint`, so each analysis can include or exclude them explicitly.

## 4. Official qualifying results (Q1 times)

In [5]:
# pull OpenF1 /session_result once per qualifying session; reuse the saved file afterwards
RESULT_FILE = f'{RAW}/openf1_session_result_qualifying.csv'

def stage_time(d, i):
    # duration is a list [Q1, Q2, Q3] in qualifying; return stage i, or None if not set
    if isinstance(d, list):
        return d[i] if len(d) > i else None
    return d if i == 0 else None

if os.path.exists(RESULT_FILE):
    temp_results_df = pd.read_csv(RESULT_FILE) # already pulled: no API calls
else:
    results_list = []
    for key in quali_df['session_key']:
        for attempt in range(3):
            try:
                response = urlopen(f'https://api.openf1.org/v1/session_result?session_key={key}')
                results_list.append(pd.DataFrame(json.loads(response.read().decode('utf-8'))))
                break
            except HTTPError as e:
                if e.code == 429: # too many requests: wait, then try again
                    time.sleep(10)
                else: # OpenF1 has no result for this session
                    print('no result for session', key)
                    break
        print(key)
        time.sleep(2.1) # stay under OpenF1's rate limit
    temp_results_df = pd.concat(results_list, ignore_index=True)
    # unpack the [Q1, Q2, Q3] list into columns before saving (a CSV can't hold lists)
    for i, stage in enumerate(['q1_s', 'q2_s', 'q3_s']):
        temp_results_df[stage] = temp_results_df['duration'].apply(lambda d: stage_time(d, i))
    temp_results_df.to_csv(RESULT_FILE, index=False)

print(len(temp_results_df), 'driver results from', temp_results_df['session_key'].nunique(), 'sessions')
temp_results_df[['session_key', 'driver_number', 'position', 'q1_s', 'q2_s', 'q3_s']].head()

7768
7775
7783
9064
9278
9074
9090
9098
9106
9112
9282
9122
9129
9135
9286
9145
9153
9161
9169
9215
9298
9207
9294
9177
9304
9308
9314
9193
9468
9476
9484
9492
9668
9664
9502
9498
9511
9519
9527
9535
9545
9541
9554
9562
9570
9578
9586
9594
9602
9612
9608
9621
9631
9627
9640
9650
9646
9658
9689
9989
9994
10002
10010
10018
10024
10029
9983
9975
9967
9959
9951
9943
9930
9935
9924
9916
9908
9900
9892
9879
9884
9873
9860
9865
9854
9841
9846
9835
1757 driver results from 88 sessions


,session_key,driver_number,position,q1_s,q2_s,q3_s
0,7768,1,1.0,91.295,90.503,89.708
1,7768,11,2.0,91.479,90.746,89.846
2,7768,16,3.0,91.094,90.282,90.000
3,7768,55,4.0,90.993,90.515,90.154
4,7768,14,5.0,91.158,90.645,90.336


This is the only new API pull. OpenF1's [`/session_result`](https://openf1.org/docs/#session-result) endpoint gives each driver's **official** best time in each qualifying stage as a three-item list `[Q1, Q2, Q3]`. We split that list into three columns (`q1_s`, `q2_s`, `q3_s`) and save the result to `data/raw/` so the pull only happens once. Because these are official results, laps deleted for track limits should already be excluded; we could not confirm this in OpenF1's documentation.

## 5. Stable team IDs across rebrands

In [6]:
# one ID per constructor, so a renamed team keeps the same dummy variable across seasons
TEAM_ID = {
    'Red Bull Racing': 'red_bull', 'Ferrari': 'ferrari', 'Mercedes': 'mercedes',
    'McLaren': 'mclaren', 'Aston Martin': 'aston_martin', 'Alpine': 'alpine',
    'Williams': 'williams', 'Haas F1 Team': 'haas',
    'AlphaTauri': 'racing_bulls', 'RB': 'racing_bulls', 'Racing Bulls': 'racing_bulls',
    'Alfa Romeo': 'sauber', 'Kick Sauber': 'sauber', 'Sauber': 'sauber',
}

# each driver's team in each session (handles mid-season transfers, e.g. Tsunoda in 2025)
session_teams = (temp_drivers_df[['session_key', 'driver_number', 'team_name']]
                 .drop_duplicates(['session_key', 'driver_number']))
session_teams['team_id'] = session_teams['team_name'].map(TEAM_ID)

unmapped = session_teams.loc[session_teams['team_id'].isna(), 'team_name'].dropna().unique()
print('team names without an ID:', list(unmapped)) # should be empty
session_teams.drop_duplicates('team_name')[['team_name', 'team_id']].sort_values('team_id')

team names without an ID: ['Audi', 'Cadillac']


,team_name,team_id
3,Alpine,alpine
5,Aston Martin,aston_martin
6,Ferrari,ferrari
7,Haas F1 Team,haas
2,McLaren,mclaren
13,Mercedes,mercedes
8,AlphaTauri,racing_bulls
2268,RB,racing_bulls
4726,Racing Bulls,racing_bulls
0,Red Bull Racing,red_bull


OpenF1 records each team under its name at the time, so one constructor can appear under two or three names: AlphaTauri → RB → Racing Bulls, and Alfa Romeo → Kick Sauber. We map every name to one **`team_id`** so each team keeps a single dummy variable in our regressions. We join teams by **session**, not by season, so a driver who changed teams mid-season (Tsunoda moved from Racing Bulls to Red Bull in 2025) is counted for the right team in each session. The printed list of unmapped names should be empty; if it isn't, add the new name to `TEAM_ID`.

## 6. Primary measure: gap to the fastest Q1 lap

In [7]:
# attach each driver's team, keep drivers who set a valid Q1 time
q1_driver_df = (temp_results_df[['session_key', 'driver_number', 'q1_s', 'q2_s']]
                .merge(session_teams[['session_key', 'driver_number', 'team_id']],
                       on=['session_key', 'driver_number'], how='left'))
print('driver rows with no team match:', q1_driver_df['team_id'].isna().sum())
print('driver rows with no Q1 time (dropped):', q1_driver_df['q1_s'].isna().sum())
q1_driver_df = q1_driver_df.dropna(subset=['q1_s', 'team_id'])

# one row per team per session: the faster of its two drivers' Q1 times
q1_team_df = duckdb.sql('''
    SELECT session_key, team_id, MIN(q1_s) AS q1_best_s
    FROM q1_driver_df GROUP BY session_key, team_id
''').df()
q1_team_df['q1_fastest_s'] = q1_team_df.groupby('session_key')['q1_best_s'].transform('min')
q1_team_df['q1_gap_pct'] = 100 * (q1_team_df['q1_best_s'] - q1_team_df['q1_fastest_s']) / q1_team_df['q1_fastest_s']
q1_team_df.head()

driver rows with no team match: 0
driver rows with no Q1 time (dropped): 17


,session_key,team_id,q1_best_s,q1_fastest_s,q1_gap_pct
0,7768,mclaren,91.652,90.993,0.724
1,7775,alpine,89.707,88.761,1.066
2,7783,red_bull,77.384,77.384,0.000
3,9064,mercedes,102.073,101.269,0.794
4,9064,haas,102.755,101.269,1.467


Following our mentor's suggestion, our **primary** comparison is each team's fastest valid Q1 lap against the fastest valid Q1 lap of that session:

> `q1_gap_pct` = (team's best Q1 lap − fastest Q1 lap) / fastest Q1 lap × 100

Q1 is the only stage every car runs, so no team is penalized for being knocked out before the track gets faster. A driver with no Q1 time (crash, failure to start, or all laps deleted) is dropped; the team then uses its other driver. The two printed counts show how many driver rows this affects.

## 7. Secondary measure: whole-session gap to pole

In [8]:
# timed laps only: drop pit-out laps and laps with no recorded time
laps_df = temp_laps_df[(temp_laps_df['is_pit_out_lap'] != True) & temp_laps_df['lap_duration'].notna()]
laps_df = laps_df.merge(session_teams[['session_key', 'driver_number', 'team_id']],
                        on=['session_key', 'driver_number'])

# team best lap of the whole session, and its % gap to the session's fastest lap (pole)
session_team_df = duckdb.sql('''
    SELECT session_key, team_id, MIN(lap_duration) AS best_lap_s
    FROM laps_df GROUP BY session_key, team_id
''').df()
session_team_df['pole_s'] = session_team_df.groupby('session_key')['best_lap_s'].transform('min')
session_team_df['gap_to_pole_pct'] = (100 * (session_team_df['best_lap_s'] - session_team_df['pole_s'])
                                      / session_team_df['pole_s'])
session_team_df.head()

,session_key,team_id,best_lap_s,pole_s,gap_to_pole_pct
0,9278,haas,103.806,101.697,2.074
1,9278,mercedes,102.252,101.697,0.546
2,9074,red_bull,86.814,86.814,0.000
3,9098,mclaren,72.776,72.272,0.697
4,9106,sauber,81.821,78.725,3.933


We keep our Phase 2 measure, the team's fastest lap of the **whole** session compared with pole, as a robustness check. It uses every timed lap (pit-out laps and untimed laps removed). It is still biased against teams knocked out in Q1, which is why it is no longer our primary measure.

## 8. Weather during Q1 and the wet-session flag

In [18]:
# when did Q1 end? the last lap finished by a driver knocked out in Q1 (Q1 time but no Q2 time)
laps_t = temp_laps_df[['session_key', 'driver_number', 'date_start', 'lap_duration']].copy()
laps_t['lap_end'] = to_est(laps_t['date_start']) + pd.to_timedelta(laps_t['lap_duration'].fillna(120), unit='s')
knocked_out = temp_results_df.loc[temp_results_df['position'] > 15, ['session_key', 'driver_number']]
q1_end = (laps_t.merge(knocked_out, on=['session_key', 'driver_number'])
          .groupby('session_key')['lap_end'].max().rename('q1_end').reset_index())

# Q1 window = session start to Q1 end; fall back to the scheduled length (18 min, 12 for sprint)
q1_window = quali_df[['session_key', 'date_start', 'is_sprint']].merge(q1_end, on='session_key', how='left')
fallback = q1_window['date_start'] + pd.to_timedelta(q1_window['is_sprint'].map({True: 12, False: 18}), unit='m')
print('sessions using the scheduled-length fallback:', q1_window['q1_end'].isna().sum())
q1_window['q1_end'] = q1_window['q1_end'].fillna(fallback)

sessions using the scheduled-length fallback: 1


The mentor asked us to **match the weather to the qualifying stage we analyze**. OpenF1 does not label which laps belong to Q1, so we estimate when Q1 ended: it is the last lap completed by any driver who was knocked out in Q1 (a Q1 time but no Q2 time), since those drivers only ran in Q1. If a session has no lap data, we use Q1's scheduled length (18 minutes, or 12 for sprint qualifying). Red-flag delays are captured automatically by the lap-based estimate, but not by the fallback.

In [19]:
# average trackside conditions inside each session's Q1 window
f1_weather = temp_f1_weather_df.copy()
f1_weather['date'] = to_est(f1_weather['date'])

f1_q1_weather_df = duckdb.sql('''
    SELECT w.session_key,
           AVG(w.track_temperature) AS track_temp_q1,
           AVG(w.air_temperature) AS air_temp_f1_q1,
           MAX(CAST(w.rainfall AS DOUBLE)) > 0 AS wet_q1,
           COUNT(*) AS n_readings_q1
    FROM f1_weather w JOIN q1_window q ON w.session_key = q.session_key
    WHERE w.date BETWEEN q.date_start AND q.q1_end
    GROUP BY w.session_key
''').df()

# same flag over the whole session, so we can see sessions where rain came after Q1
f1_session_weather_df = duckdb.sql('''
    SELECT w.session_key, AVG(w.track_temperature) AS track_temp,
           MAX(CAST(w.rainfall AS DOUBLE)) > 0 AS rain_any
    FROM f1_weather w JOIN quali_df q ON w.session_key = q.session_key
    WHERE w.date BETWEEN q.date_start AND q.date_end
    GROUP BY w.session_key
''').df()

print('sessions wet during Q1:', int(f1_q1_weather_df['wet_q1'].sum()),
      '| wet at any point in the session:', int(f1_session_weather_df['rain_any'].sum()))
f1_q1_weather_df.describe().loc[['count', 'mean', 'min', 'max']]

sessions wet during Q1: 9 | wet at any point in the session: 12


,session_key,track_temp_q1,air_temp_f1_q1,n_readings_q1
count,87.000,87.000,87.000,87.000
mean,9541.391,35.226,24.233,24.126
min,7768.000,12.633,11.729,12.000
max,10029.000,50.052,35.700,70.000


In [21]:
# there seems to be sessions with an weirdly long Q1 window (normal is about 18-25 readings), checking
long_q1 = f1_q1_weather_df[f1_q1_weather_df['n_readings_q1'] > 30].merge(
    quali_df[['session_key', 'year', 'circuit_short_name', 'session_name']], on='session_key')
long_q1[['year', 'circuit_short_name', 'session_name', 'n_readings_q1', 'wet_q1']]

,year,circuit_short_name,session_name,n_readings_q1,wet_q1
0,2023,Interlagos,Qualifying,40,False
1,2025,Imola,Qualifying,40,False
2,2025,Miami,Qualifying,35,False
3,2025,Silverstone,Qualifying,40,False
4,2023,Spa-Francorchamps,Qualifying,35,False
5,2025,Sakhir,Qualifying,51,False
6,2025,Austin,Qualifying,32,False
7,2023,Silverstone,Qualifying,40,True
8,2024,Hungaroring,Qualifying,36,True
9,2023,Monte Carlo,Qualifying,31,False


Using the Q1 window, we average OpenF1's trackside readings (about one per minute) to get **`track_temp_q1`**, the track temperature during the stage our performance measure comes from. A session is **wet during Q1** (`wet_q1 = True`) if any reading in that window recorded rainfall. We also keep `rain_any` for the whole session, so we can see sessions where rain only arrived later. `n_readings_q1` should be roughly 18–25 for main qualifying and 12–15 for sprint qualifying (one reading per minute); much lower values mean the readings and the window did not line up.

## 9. Merge into the analysis-ready table

In [11]:
# circuit elevation (m above sea level) and local time zone
CIRCUIT_ELEVATION_M = {'Austin': 161, 'Baku': -15, 'Catalunya': 120, 'Hungaroring': 200, 'Imola': 49,
    'Interlagos': 785, 'Jeddah': 12, 'Las Vegas': 620, 'Lusail': 5, 'Melbourne': 10, 'Mexico City': 2238,
    'Miami': 3, 'Monte Carlo': 40, 'Montreal': 13, 'Monza': 162, 'Sakhir': 7, 'Shanghai': 5,
    'Silverstone': 153, 'Singapore': 15, 'Spa-Francorchamps': 401, 'Spielberg': 678, 'Suzuka': 45,
    'Yas Marina Circuit': 3, 'Zandvoort': 5}
CIRCUIT_TZ = {'Sakhir': 'Asia/Bahrain', 'Jeddah': 'Asia/Riyadh', 'Melbourne': 'Australia/Melbourne',
    'Baku': 'Asia/Baku', 'Miami': 'America/New_York', 'Monte Carlo': 'Europe/Monaco',
    'Catalunya': 'Europe/Madrid', 'Montreal': 'America/Toronto', 'Spielberg': 'Europe/Vienna',
    'Silverstone': 'Europe/London', 'Hungaroring': 'Europe/Budapest', 'Spa-Francorchamps': 'Europe/Brussels',
    'Zandvoort': 'Europe/Amsterdam', 'Monza': 'Europe/Rome', 'Singapore': 'Asia/Singapore',
    'Suzuka': 'Asia/Tokyo', 'Lusail': 'Asia/Qatar', 'Austin': 'America/Chicago',
    'Mexico City': 'America/Mexico_City', 'Interlagos': 'America/Sao_Paulo',
    'Las Vegas': 'America/Los_Angeles', 'Yas Marina Circuit': 'Asia/Dubai',
    'Shanghai': 'Asia/Shanghai', 'Imola': 'Europe/Rome'}

# Open-Meteo: one row per session (air temperature and cloud cover at the start hour)
openmeteo_df = temp_openmeteo_df[['session_key', 'temperature_2m', 'cloud_cover', 'precipitation']]
assert openmeteo_df['session_key'].is_unique, 'Open-Meteo has more than one row for a session'

# most recent team name for each team_id (for readable labels)
team_names = session_teams.dropna(subset=['team_id']).drop_duplicates('team_id', keep='last')[['team_id', 'team_name']]

These lookup tables carry over from Phase 2: circuit elevations (metres above sea level) and each circuit's time zone, which we use to decide whether a session ran at night locally. We also check that Open-Meteo has exactly one row per session, so the merge below cannot duplicate rows.

In [12]:
# start from the Q1 measure (primary) and attach everything else by session_key / team_id
analysis_df = (q1_team_df
    .merge(session_team_df, on=['session_key', 'team_id'], how='left') # secondary measure
    .merge(quali_df[['session_key', 'year', 'circuit_short_name', 'session_name', 'is_sprint', 'date_start']],
           on='session_key')
    .merge(team_names, on='team_id', how='left')
    .merge(openmeteo_df, on='session_key', how='left')
    .merge(f1_session_weather_df, on='session_key', how='left')
    .merge(f1_q1_weather_df, on='session_key', how='left'))

# circuit variables and flags (date_start is EST, so convert to the circuit's local time first)
analysis_df['elevation_m'] = analysis_df['circuit_short_name'].map(CIRCUIT_ELEVATION_M)
analysis_df['local_hour'] = [t.tz_localize('EST').tz_convert(CIRCUIT_TZ[c]).hour
                             for t, c in zip(analysis_df['date_start'], analysis_df['circuit_short_name'])]
analysis_df['is_night'] = (analysis_df['local_hour'] >= 18) | (analysis_df['local_hour'] < 5)
analysis_df['is_mexico_city'] = analysis_df['circuit_short_name'] == 'Mexico City'
analysis_df['is_mercedes'] = analysis_df['team_id'] == 'mercedes'

assert not analysis_df.duplicated(['session_key', 'team_id']).any(), 'duplicate team-session rows'
print(len(analysis_df), 'team-sessions from', analysis_df['session_key'].nunique(), 'sessions')

878 team-sessions from 88 sessions


We build one row per **team per qualifying session**, starting from the Q1 measure and attaching the whole-session measure, session details, Open-Meteo weather, and both OpenF1 weather summaries. Every join is on `session_key` (and `team_id`), so weather can only attach to the session it was measured in. The `assert` stops the notebook if any team-session appears twice.

## 10. Relative pace (recalculated season averages)

In [13]:
# season half: first or second half of each year's qualifying sessions, by date
order = analysis_df.drop_duplicates('session_key')[['session_key', 'year', 'date_start']].copy()
order['rank'] = order.groupby('year')['date_start'].rank(method='first')
order['n'] = order.groupby('year')['session_key'].transform('count')
order['season_half'] = (order['rank'] > order['n'] / 2).map({False: 1, True: 2})
analysis_df = analysis_df.merge(order[['session_key', 'season_half']], on='session_key')

# primary: Q1 gap minus the team's average Q1 gap that season
analysis_df['q1_season_avg_gap_pct'] = analysis_df.groupby(['team_id', 'year'])['q1_gap_pct'].transform('mean')
analysis_df['q1_relative_pace'] = analysis_df['q1_gap_pct'] - analysis_df['q1_season_avg_gap_pct']

# robustness: same, but averaged within each half-season (car upgrades shift the baseline)
half_avg = analysis_df.groupby(['team_id', 'year', 'season_half'])['q1_gap_pct'].transform('mean')
analysis_df['q1_relative_pace_half'] = analysis_df['q1_gap_pct'] - half_avg

# secondary: whole-session gap minus the team's average whole-session gap that season
analysis_df['season_avg_gap_pct'] = analysis_df.groupby(['team_id', 'year'])['gap_to_pole_pct'].transform('mean')
analysis_df['relative_pace'] = analysis_df['gap_to_pole_pct'] - analysis_df['season_avg_gap_pct']

analysis_df[['q1_relative_pace', 'q1_relative_pace_half', 'relative_pace']].describe().round(3)

,q1_relative_pace,q1_relative_pace_half,relative_pace
count,878.000,878.000,868.000
mean,0.000,-0.000,0.000
std,0.432,0.424,1.350
min,-1.056,-1.091,-2.530
25%,-0.260,-0.247,-0.438
50%,-0.059,-0.063,-0.110
75%,0.191,0.188,0.194
max,4.520,4.400,13.294


**Relative pace** is a team's gap this weekend minus its own average gap that season, so it measures whether a team was quicker (negative) or slower (positive) **than usual**, not who is fastest overall.

- **`q1_relative_pace`** (primary) uses the Q1 gap, with season averages **recalculated** on the Q1 measure, as the mentor asked.
- **`q1_relative_pace_half`** averages within each half-season instead, because mid-season upgrades can shift a team's baseline. If our results change when we swap this in, car development is likely driving them.
- **`relative_pace`** is the old whole-session version, kept as a check.

Season averages are grouped by `team_id`, so a renamed team keeps one baseline per season.

## 11. Checking the largest gaps

In [14]:
# the largest Q1 gaps, with the conditions that might explain them
analysis_df.sort_values('q1_gap_pct', ascending=False)[
    ['year', 'circuit_short_name', 'is_sprint', 'team_id', 'q1_gap_pct', 'gap_to_pole_pct', 'wet_q1', 'track_temp_q1']
].head(10)

,year,circuit_short_name,is_sprint,team_id,q1_gap_pct,gap_to_pole_pct,wet_q1,track_temp_q1
250,2024,Silverstone,False,alpine,5.595,10.182,True,21.588
425,2024,Interlagos,False,haas,3.058,9.381,True,22.690
473,2024,Silverstone,False,haas,2.660,0.605,True,21.588
119,2023,Baku,True,racing_bulls,2.544,3.677,False,48.114
777,2023,Spa-Francorchamps,True,haas,2.492,11.025,True,22.533
586,2024,Interlagos,False,sauber,2.385,3.677,True,22.690
65,2023,Spa-Francorchamps,True,sauber,2.384,10.907,True,22.533
228,2023,Montreal,False,racing_bulls,2.344,5.108,True,18.429
54,2023,Baku,True,sauber,2.292,3.422,False,48.114
801,2024,Interlagos,True,aston_martin,2.160,3.017,False,39.517


The mentor flagged a 16.3% whole-session gap in Phase 2. This table lists the ten largest Q1 gaps alongside the whole-session gap, whether Q1 was wet, and track temperature, so we can check each one against what happened in that session (rain, a red flag, a crash, or a car that set only a slow lap).

**To fill in after running:** _for each session above, note the likely cause and whether it is kept. Our rule (section 13) is to keep every dry session and to exclude wet-Q1 sessions from the primary models, so no team-session is dropped only because its gap is large._

## 12. Export `analysis_ready.csv`

In [15]:
# final column order: identifiers, primary measure, secondary measure, weather, circuit flags
COLUMNS = ['session_key', 'year', 'circuit_short_name', 'session_name', 'is_sprint', 'season_half',
           'team_id', 'team_name', 'is_mercedes', 'date_start',
           'q1_best_s', 'q1_fastest_s', 'q1_gap_pct', 'q1_season_avg_gap_pct', 'q1_relative_pace',
           'q1_relative_pace_half',
           'best_lap_s', 'pole_s', 'gap_to_pole_pct', 'season_avg_gap_pct', 'relative_pace',
           'track_temp_q1', 'air_temp_f1_q1', 'wet_q1', 'n_readings_q1', 'track_temp', 'rain_any',
           'temperature_2m', 'cloud_cover', 'precipitation',
           'elevation_m', 'local_hour', 'is_night', 'is_mexico_city']

analysis_ready = (analysis_df[COLUMNS].rename(columns={'date_start': 'date_start_est'})
                  .sort_values(['date_start_est', 'team_id']))
analysis_ready.to_csv('analysis_ready.csv', index=False)

print(analysis_ready.shape[0], 'rows x', analysis_ready.shape[1], 'columns saved to analysis_ready.csv')
print('missing values per column (only columns with any):')
print(analysis_ready.isna().sum()[lambda s: s > 0].to_string())

878 rows x 34 columns saved to analysis_ready.csv
missing values per column (only columns with any):
best_lap_s         10
pole_s             10
gap_to_pole_pct    10
relative_pace      10
track_temp_q1      10
air_temp_f1_q1     10
wet_q1             10
n_readings_q1      10


We save the analysis-ready table. The printout reports how many rows and columns were saved and which columns have missing values. Expected gaps: whole-session columns are missing for any session OpenF1 has no lap data for, and OpenF1 weather columns are missing if a session had no trackside readings.

## 13. Cleaning rules (for the data description and preregistration)

- **Unit of observation:** one team in one qualifying session. Teams in the same session share weather and the same fastest-lap reference, and circuits repeat across seasons, so the 870-odd rows are **not independent observations**. Our models should account for this, for example with standard errors clustered by session.
- **Primary outcome:** `q1_relative_pace`. **Robustness checks:** `q1_relative_pace_half` (half-season baseline) and `relative_pace` (whole-session gap to pole).
- **Teams:** identified by `team_id` (AlphaTauri / RB / Racing Bulls → `racing_bulls`; Alfa Romeo / Kick Sauber → `sauber`). Drivers are assigned to teams per session.
- **Missing laps:** a driver with no official Q1 time is dropped and the team uses its other driver; a team with no Q1 time from either driver is dropped for that session.
- **Wet sessions:** sessions with `wet_q1 = True` are **excluded from the primary models** and reported separately. Sessions where rain fell only after Q1 (`rain_any = True`, `wet_q1 = False`) are kept.
- **Sprint qualifying:** kept and flagged with `is_sprint`; each analysis states whether it includes them.
- **Weather:** `track_temp_q1` (OpenF1, trackside, Q1 window) is the main temperature measure; `temperature_2m` and `cloud_cover` (Open-Meteo, session-start hour) are used for cloud cover and as a check on air temperature.
- **Large gaps:** kept unless the session is wet in Q1; the largest are listed in section 11 with their likely causes.